# V2 Ising Model Analysis

Comprehensive analysis of 3D Ising model Monte Carlo simulations.
Covers validation, finite-size scaling, J-fitting, Kibble-Zurek, and domain coarsening.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import curve_fit, minimize_scalar
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Add parent dir so we can import pub_style
sys.path.insert(0, str(Path('.').resolve().parent))
from pub_style import setup_style, COLORS, SIZE_MARKERS, label_panel, jackknife_error

setup_style()
import matplotlib.pyplot as plt

DATADIR = Path('data')
FIGDIR = Path('figures')
FIGDIR.mkdir(exist_ok=True)

# 3D Ising exact values (literature)
TC_3D = 4.5115232  # Tc/J for simple cubic
NU_3D = 0.6301
GAMMA_3D = 1.2372
BETA_3D = 0.3265
ALPHA_3D = 0.1096

# 2D Ising exact values (Onsager)
TC_2D = 2.0 / np.log(1 + np.sqrt(2))  # = 2.26919...

print(f"Data directory: {DATADIR.resolve()}")
print(f"2D Onsager Tc = {TC_2D:.5f}")
print(f"3D Cubic  Tc = {TC_3D:.7f}")

In [ ]:
def load_fss(subdir, prefix='fss'):
    """Load all FSS CSV files from a subdirectory, return dict {N: DataFrame}."""
    data = {}
    for f in sorted(DATADIR.joinpath(subdir).glob(f'{prefix}_N*.csv')):
        N = int(f.stem.split('N')[1])
        data[N] = pd.read_csv(f)
    print(f"Loaded {len(data)} sizes from {subdir}/: {sorted(data.keys())}")
    return data

def load_raw(subdir='fss_raw'):
    """Load raw per-sample CSV files, return dict {N: DataFrame}."""
    data = {}
    for f in sorted(DATADIR.joinpath(subdir).glob('fss_raw_N*.csv')):
        N = int(f.stem.split('N')[1])
        data[N] = pd.read_csv(f)
    print(f"Loaded {len(data)} raw sizes from {subdir}/: {sorted(data.keys())}")
    return data

def binder(m2, m4):
    """Binder cumulant U = 1 - <m^4>/(3<m^2>^2)."""
    return 1.0 - m4 / (3.0 * m2**2)

def chi_connected(T, M, M2, N):
    """Connected susceptibility: chi = beta * V * (<|m|^2> - <|m|>^2)."""
    beta = 1.0 / T
    V = N**3
    return beta * V * (M2 - M**2)

---
## 1. Validation

### 1a. 2D Onsager Solution

Compare our MC results on the 2D square lattice against the exact Onsager solution:
- $T_c = 2/\ln(1+\sqrt{2}) \approx 2.269$
- $E(T)$, $M(T)$ curves should match known analytic forms
- At $T \to 0$: $E \to -2J$, $|m| \to 1$
- At $T \to \infty$: $E \to 0$, $|m| \to 0$

In [ ]:
# Load 2D validation data
val = load_fss('validation')

# Onsager exact energy (per spin) for 2D square lattice
def onsager_energy(T, J=1.0):
    """Exact internal energy per spin for 2D Ising (Onsager)."""
    beta = J / T
    k = 2 * np.sinh(2 * beta) / np.cosh(2 * beta)**2
    # Complete elliptic integral of the first kind
    from scipy.special import ellipk
    K1 = ellipk(k**2)
    return -J * (1.0 / np.tanh(2 * beta)) * (1 + (2.0 / np.pi) * (2 * np.tanh(2 * beta)**2 - 1) * K1)

# Onsager exact magnetisation (per spin) for T < Tc
def onsager_magnetisation(T, J=1.0):
    """Exact spontaneous magnetisation for 2D Ising (Yang, 1952)."""
    beta = J / T
    if T >= TC_2D:
        return 0.0
    return (1 - np.sinh(2 * beta)**(-4))**(1/8)

onsager_mag_vec = np.vectorize(onsager_magnetisation)

# Plot 2D validation
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

T_fine = np.linspace(1.5, 3.5, 200)

# (a) Energy
ax = axes[0]
for N in sorted(val.keys()):
    if N >= 32:  # 2D sizes
        df = val[N]
        if df['E'].iloc[0] > -1.5:  # 2D: E ~ -2 at low T
            continue
        ax.plot(df['T'], df['E'], SIZE_MARKERS.get(N, 'o'), ms=4, label=f'L={N}')
ax.plot(T_fine, [onsager_energy(t) for t in T_fine], 'k-', lw=1, label='Onsager exact')
ax.axvline(TC_2D, color='gray', ls='--', lw=0.5, alpha=0.5)
ax.set_xlabel('T / J')
ax.set_ylabel('E / J (per spin)')
ax.legend(fontsize=6)
label_panel(ax, 'a')

# (b) Magnetisation
ax = axes[1]
for N in sorted(val.keys()):
    if N >= 32:
        df = val[N]
        if df['E'].iloc[0] > -1.5:
            continue
        ax.plot(df['T'], df['M'], SIZE_MARKERS.get(N, 'o'), ms=4, label=f'L={N}')
ax.plot(T_fine, onsager_mag_vec(T_fine), 'k-', lw=1, label='Onsager exact')
ax.axvline(TC_2D, color='gray', ls='--', lw=0.5, alpha=0.5)
ax.set_xlabel('T / J')
ax.set_ylabel('|m| (per spin)')
ax.legend(fontsize=6)
label_panel(ax, 'b')

# (c) Binder cumulant
ax = axes[2]
for N in sorted(val.keys()):
    if N >= 32:
        df = val[N]
        if df['E'].iloc[0] > -1.5:
            continue
        U = binder(df['M2'], df['M4'])
        ax.plot(df['T'], U, SIZE_MARKERS.get(N, 'o')+'-', ms=4, lw=0.8, label=f'L={N}')
ax.axvline(TC_2D, color='gray', ls='--', lw=0.5, label=f'$T_c$ = {TC_2D:.4f}')
ax.axhline(2/3, color='gray', ls=':', lw=0.5)
ax.set_xlabel('T / J')
ax.set_ylabel('Binder cumulant $U_4$')
ax.legend(fontsize=6)
label_panel(ax, 'c')

fig.suptitle('2D Onsager Validation', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(FIGDIR / 'fig1_2d_onsager_validation.pdf')
plt.show()
print("PASS: 2D Onsager validation plotted.")

### 1b. 3D Known Limits

Verify the 3D cubic lattice simulation at extreme temperatures:
- $T \to 0$: $E/N \to -3J$ (ground state, $z=6$), $|m| \to 1$
- $T \to \infty$: $E/N \to 0$, $|m| \to 0$, $C_v \to 0$

In [ ]:
# 3D validation: check known limits
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

for N in sorted(val.keys()):
    df = val[N]
    if df['E'].iloc[0] < -2.5:  # 3D data (E ~ -3 at low T)
        c = COLORS[sorted(val.keys()).index(N) % len(COLORS)]
        mk = SIZE_MARKERS.get(N, 'o')
        axes[0].plot(df['T'], df['E'], mk, ms=4, color=c, label=f'L={N}')
        axes[1].plot(df['T'], df['M'], mk, ms=4, color=c, label=f'L={N}')
        U = binder(df['M2'], df['M4'])
        axes[2].plot(df['T'], U, mk+'-', ms=4, lw=0.8, color=c, label=f'L={N}')

# Annotate limits
axes[0].axhline(-3.0, color='gray', ls=':', lw=0.5, label='$E=-3J$ (ground)')
axes[0].axhline(0.0, color='gray', ls=':', lw=0.5)
axes[0].axvline(TC_3D, color='red', ls='--', lw=0.5, alpha=0.7, label=f'$T_c={TC_3D:.3f}$')
axes[0].set_xlabel('T / J'); axes[0].set_ylabel('E / J (per spin)')
axes[0].legend(fontsize=6)
label_panel(axes[0], 'a')

axes[1].axhline(1.0, color='gray', ls=':', lw=0.5)
axes[1].axhline(0.0, color='gray', ls=':', lw=0.5)
axes[1].axvline(TC_3D, color='red', ls='--', lw=0.5, alpha=0.7)
axes[1].set_xlabel('T / J'); axes[1].set_ylabel('|m| (per spin)')
axes[1].legend(fontsize=6)
label_panel(axes[1], 'b')

axes[2].axvline(TC_3D, color='red', ls='--', lw=0.5, alpha=0.7)
axes[2].axhline(2/3, color='gray', ls=':', lw=0.5)
axes[2].set_xlabel('T / J'); axes[2].set_ylabel('$U_4$')
axes[2].legend(fontsize=6)
label_panel(axes[2], 'c')

fig.suptitle('3D Known Limits Validation', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(FIGDIR / 'fig2_3d_limits_validation.pdf')
plt.show()

# Quantitative checks
for N in sorted(val.keys()):
    df = val[N]
    if df['E'].iloc[0] < -2.5:
        E_low = df['E'].iloc[0]
        M_low = df['M'].iloc[0]
        E_high = df['E'].iloc[-1]
        M_high = df['M'].iloc[-1]
        print(f"L={N:3d}: E(T_min)={E_low:.4f} (expect -3.0), |m|(T_min)={M_low:.4f} (expect 1.0)")
        print(f"        E(T_max)={E_high:.4f} (expect ~0),    |m|(T_max)={M_high:.4f} (expect ~0)")

---
## 2. Finite-Size Scaling (3D Cubic)

The heart of the analysis. We extract $T_c$ and critical exponents $\nu$, $\gamma/\nu$, $\beta/\nu$ from:
1. **Binder cumulant crossing** $\to T_c$
2. **Susceptibility peak scaling** $\chi_{\max} \sim L^{\gamma/\nu}$ $\to \gamma/\nu$
3. **Magnetisation at $T_c$** $M(T_c) \sim L^{-\beta/\nu}$ $\to \beta/\nu$
4. **Scaling collapse** with optimised $(\nu, \gamma/\nu)$

### 2a. Raw observables

In [ ]:
# Load 3D FSS data
fss = load_fss('fss')

fig, axes = plt.subplots(2, 2, figsize=(8, 6))

for N in sorted(fss.keys()):
    df = fss[N]
    c = COLORS[sorted(fss.keys()).index(N) % len(COLORS)]
    mk = SIZE_MARKERS.get(N, 'o')
    kw = dict(marker=mk, ms=3, color=c, lw=0.8, label=f'L={N}')
    
    # Compute connected chi
    V = N**3
    chi_conn = (1.0 / df['T']) * V * (df['M2'] - df['M']**2)
    U = binder(df['M2'], df['M4'])
    
    axes[0,0].plot(df['T'], df['E'], **kw)
    axes[0,1].plot(df['T'], df['M'], **kw)
    axes[1,0].plot(df['T'], chi_conn, **kw)
    axes[1,1].plot(df['T'], U, **kw)

for ax in axes.flat:
    ax.axvline(TC_3D, color='red', ls='--', lw=0.5, alpha=0.5)

axes[0,0].set_ylabel('E / J'); label_panel(axes[0,0], 'a')
axes[0,1].set_ylabel('|m|'); label_panel(axes[0,1], 'b')
axes[1,0].set_ylabel('$\\chi_{\\mathrm{conn}}$'); label_panel(axes[1,0], 'c')
axes[1,1].set_ylabel('$U_4$'); label_panel(axes[1,1], 'd')
for ax in axes[1,:]:
    ax.set_xlabel('T / J')
axes[0,1].legend(fontsize=6, loc='upper right')

fig.suptitle('3D Cubic Ising: Raw Observables (Wolff)', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(FIGDIR / 'fig3_fss_raw_observables.pdf')
plt.show()

### 2b. Binder Cumulant Crossing for $T_c$

The Binder cumulant $U_4 = 1 - \langle m^4 \rangle / (3 \langle m^2 \rangle^2)$ is scale-invariant at $T_c$.
Crossing points of $U_4(T)$ for different $L$ converge to $T_c$ as $L \to \infty$.

In [ ]:
# Binder cumulant crossing analysis
from scipy.interpolate import interp1d

sizes = sorted(fss.keys())
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

# (a) Binder cumulant zoom near Tc
ax = axes[0]
binder_interps = {}
for N in sizes:
    df = fss[N]
    U = binder(df['M2'], df['M4'])
    c = COLORS[sizes.index(N) % len(COLORS)]
    mk = SIZE_MARKERS.get(N, 'o')
    ax.plot(df['T'], U, mk+'-', ms=3, lw=0.8, color=c, label=f'L={N}')
    # Interpolator for crossing detection
    binder_interps[N] = interp1d(df['T'].values, U.values, kind='cubic', fill_value='extrapolate')

ax.axvline(TC_3D, color='red', ls='--', lw=0.5, alpha=0.7, label=f'$T_c$ = {TC_3D:.4f}')
ax.axhline(2/3, color='gray', ls=':', lw=0.5)
ax.set_xlim(4.0, 5.0)
ax.set_xlabel('T / J')
ax.set_ylabel('$U_4$')
ax.legend(fontsize=6)
label_panel(ax, 'a')

# Find pairwise crossings
crossings = []
from scipy.optimize import brentq
for i in range(len(sizes)):
    for j in range(i+1, len(sizes)):
        L1, L2 = sizes[i], sizes[j]
        f1, f2 = binder_interps[L1], binder_interps[L2]
        try:
            diff = lambda T: f1(T) - f2(T)
            Tc_cross = brentq(diff, 4.0, 5.0)
            crossings.append((L1, L2, Tc_cross))
        except ValueError:
            pass

# (b) Crossing points vs 1/L_min
ax = axes[1]
if crossings:
    for L1, L2, Tc in crossings:
        ax.plot(1/min(L1, L2), Tc, 'ko', ms=4)
        ax.annotate(f'{L1},{L2}', (1/min(L1, L2), Tc), fontsize=5, 
                   textcoords='offset points', xytext=(3, 3))
    
    Tc_avg = np.mean([c[2] for c in crossings])
    Tc_std = np.std([c[2] for c in crossings])
    ax.axhline(TC_3D, color='red', ls='--', lw=0.5, label=f'Exact $T_c$ = {TC_3D:.4f}')
    ax.axhline(Tc_avg, color='blue', ls=':', lw=0.5, label=f'Mean = {Tc_avg:.4f}')
    ax.set_xlabel('$1/L_{\\min}$')
    ax.set_ylabel('$T_c$ crossing')
    ax.legend(fontsize=6)
    label_panel(ax, 'b')
    
    print(f"\nBinder crossing results:")
    for L1, L2, Tc in crossings:
        print(f"  L={L1:2d} x L={L2:2d}: Tc = {Tc:.4f}")
    print(f"\n  Mean Tc = {Tc_avg:.4f} +/- {Tc_std:.4f}")
    print(f"  Exact Tc = {TC_3D:.4f}")
    print(f"  Error: {abs(Tc_avg - TC_3D)/TC_3D*100:.2f}%")

plt.tight_layout()
plt.savefig(FIGDIR / 'fig4_binder_crossing.pdf')
plt.show()

### 2c. Histogram Reweighting (Ferrenberg-Swendsen)

Interpolate observables to a fine temperature grid near $T_c$ using single-histogram reweighting
from the raw per-sample energy/magnetisation time series.

In [ ]:
# Histogram reweighting
raw = load_raw('fss_raw')

def reweight_single(e_samples, m_samples, beta_sim, beta_target, V):
    """Single-histogram reweighting from beta_sim to beta_target.
    
    e_samples: energy per spin array
    m_samples: |m| per spin array
    beta_sim: inverse temperature of simulation
    beta_target: target inverse temperature
    V: number of spins (N^3)
    """
    dE = (beta_target - beta_sim) * V * e_samples  # total energy shift
    # Numerical stability: subtract max
    dE_shifted = dE - np.max(dE)
    w = np.exp(-dE_shifted)
    Z = np.sum(w)
    
    m_avg = np.sum(w * np.abs(m_samples)) / Z
    m2_avg = np.sum(w * m_samples**2) / Z
    m4_avg = np.sum(w * m_samples**4) / Z
    e_avg = np.sum(w * e_samples) / Z
    e2_avg = np.sum(w * e_samples**2) / Z
    
    chi_conn = beta_target * V * (m2_avg - m_avg**2)
    Cv = beta_target**2 * V * (e2_avg - e_avg**2)
    U = 1.0 - m4_avg / (3.0 * m2_avg**2)
    
    return dict(T=1/beta_target, E=e_avg, M=m_avg, M2=m2_avg, M4=m4_avg,
                chi=chi_conn, Cv=Cv, U=U)

def reweight_patchwork(raw_df, N, T_grid):
    """Patchwork reweighting: for each target T, use the nearest simulation T."""
    V = N**3
    sim_temps = raw_df['T'].unique()
    
    results = []
    for T_target in T_grid:
        beta_target = 1.0 / T_target
        # Find nearest simulation temperature
        idx = np.argmin(np.abs(sim_temps - T_target))
        T_sim = sim_temps[idx]
        beta_sim = 1.0 / T_sim
        
        mask = raw_df['T'] == T_sim
        e = raw_df.loc[mask, 'e'].values  # energy per spin
        m = raw_df.loc[mask, 'm_abs'].values  # |m| per spin
        
        r = reweight_single(e, m, beta_sim, beta_target, V)
        results.append(r)
    
    return pd.DataFrame(results)

# Reweight all sizes to a fine grid
T_fine = np.linspace(4.30, 4.70, 200)
rw = {}
for N in sorted(raw.keys()):
    rw[N] = reweight_patchwork(raw[N], N, T_fine)
    print(f"  Reweighted L={N}: chi_max = {rw[N]['chi'].max():.1f} at T = {rw[N].loc[rw[N]['chi'].idxmax(), 'T']:.4f}")

# Plot reweighted observables
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

for N in sorted(rw.keys()):
    c = COLORS[sorted(rw.keys()).index(N) % len(COLORS)]
    kw = dict(color=c, lw=1.0, label=f'L={N}')
    axes[0].plot(rw[N]['T'], rw[N]['chi'], **kw)
    axes[1].plot(rw[N]['T'], rw[N]['M'], **kw)
    axes[2].plot(rw[N]['T'], rw[N]['U'], **kw)

for ax in axes:
    ax.axvline(TC_3D, color='red', ls='--', lw=0.5, alpha=0.5)

axes[0].set_ylabel('$\\chi_{\\mathrm{conn}}$'); axes[0].set_xlabel('T / J')
axes[1].set_ylabel('|m|'); axes[1].set_xlabel('T / J')
axes[2].set_ylabel('$U_4$'); axes[2].set_xlabel('T / J')
axes[2].axhline(2/3, color='gray', ls=':', lw=0.5)
label_panel(axes[0], 'a'); label_panel(axes[1], 'b'); label_panel(axes[2], 'c')
axes[0].legend(fontsize=6)

fig.suptitle('Histogram Reweighting (Ferrenberg-Swendsen)', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(FIGDIR / 'fig5_histogram_reweighting.pdf')
plt.show()

### 2d. Critical Exponent Extraction

Extract exponents from peak scaling with jackknife error bars:
- $\chi_{\max}(L) \sim L^{\gamma/\nu}$ (log-log fit)
- $M(T_c, L) \sim L^{-\beta/\nu}$ (log-log fit)
- $dU/dT|_{\max}(L) \sim L^{1/\nu}$ (log-log fit)

In [ ]:
# Extract chi_max, M(Tc), dU/dT_max from reweighted data
# Use Tc from our Binder crossing if available, else literature value
Tc_use = Tc_avg if crossings else TC_3D
print(f"Using Tc = {Tc_use:.4f} for exponent extraction\n")

exponent_data = []
for N in sorted(rw.keys()):
    df_rw = rw[N]
    V = N**3
    
    # chi_max from reweighted
    chi_max = df_rw['chi'].max()
    
    # M at Tc (interpolate)
    m_interp = interp1d(df_rw['T'], df_rw['M'], kind='cubic')
    M_Tc = float(m_interp(Tc_use)) if df_rw['T'].min() < Tc_use < df_rw['T'].max() else np.nan
    
    # dU/dT max (numerical derivative)
    dT = df_rw['T'].diff().median()
    dU_dT = np.gradient(df_rw['U'].values, df_rw['T'].values)
    dU_dT_min = np.min(dU_dT)  # steepest negative slope
    
    exponent_data.append(dict(L=N, chi_max=chi_max, M_Tc=M_Tc, dU_dT_min=abs(dU_dT_min)))
    print(f"  L={N:3d}: chi_max={chi_max:.2f}, M(Tc)={M_Tc:.4f}, |dU/dT|_max={abs(dU_dT_min):.2f}")

exp_df = pd.DataFrame(exponent_data)

# Jackknife error on exponents from raw data
def jackknife_chi_max(raw_df, N, T_grid, n_blocks=10):
    """Jackknife estimate of chi_max error."""
    V = N**3
    sim_temps = raw_df['T'].unique()
    n_samples = raw_df.groupby('T').size().min()
    block_size = n_samples // n_blocks
    if block_size < 10:
        return 0.0
    
    chi_maxes = []
    for b in range(n_blocks):
        # Leave out block b
        results = []
        for T_target in T_grid:
            beta_target = 1.0 / T_target
            idx = np.argmin(np.abs(sim_temps - T_target))
            T_sim = sim_temps[idx]
            beta_sim = 1.0 / T_sim
            mask = raw_df['T'] == T_sim
            e_all = raw_df.loc[mask, 'e'].values
            m_all = raw_df.loc[mask, 'm_abs'].values
            # Drop block b
            keep = np.ones(len(e_all), dtype=bool)
            keep[b*block_size:(b+1)*block_size] = False
            e, m = e_all[keep], m_all[keep]
            r = reweight_single(e, m, beta_sim, beta_target, V)
            results.append(r)
        df_jk = pd.DataFrame(results)
        chi_maxes.append(df_jk['chi'].max())
    
    chi_maxes = np.array(chi_maxes)
    full = np.mean(chi_maxes)
    var = (n_blocks - 1) / n_blocks * np.sum((chi_maxes - full)**2)
    return np.sqrt(var)

# Get jackknife errors
chi_errors = {}
for N in sorted(raw.keys()):
    chi_errors[N] = jackknife_chi_max(raw[N], N, T_fine, n_blocks=10)
    print(f"  L={N}: chi_max error = {chi_errors[N]:.2f}")

exp_df['chi_max_err'] = exp_df['L'].map(chi_errors).fillna(0)

In [ ]:
# Log-log fits for exponents
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

# Filter out NaN
valid = exp_df.dropna()

# (a) gamma/nu from chi_max ~ L^(gamma/nu)
ax = axes[0]
logL = np.log(valid['L'])
logChi = np.log(valid['chi_max'])
p, cov = np.polyfit(logL, logChi, 1, cov=True)
gamma_nu = p[0]
gamma_nu_err = np.sqrt(cov[0, 0])

ax.errorbar(valid['L'], valid['chi_max'], yerr=valid['chi_max_err'],
            fmt='ko', ms=5, capsize=3)
L_fit = np.linspace(valid['L'].min()*0.8, valid['L'].max()*1.2, 50)
ax.plot(L_fit, np.exp(p[1]) * L_fit**p[0], 'r-', lw=1,
        label=f'$\\gamma/\\nu$ = {gamma_nu:.3f}({int(gamma_nu_err*1000):d})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('L'); ax.set_ylabel('$\\chi_{\\max}$')
ax.legend(fontsize=7)
label_panel(ax, 'a')

# (b) beta/nu from M(Tc) ~ L^(-beta/nu)
ax = axes[1]
logM = np.log(valid['M_Tc'])
p2, cov2 = np.polyfit(logL, logM, 1, cov=True)
beta_nu = -p2[0]  # negative slope
beta_nu_err = np.sqrt(cov2[0, 0])

ax.plot(valid['L'], valid['M_Tc'], 'ko', ms=5)
ax.plot(L_fit, np.exp(p2[1]) * L_fit**p2[0], 'r-', lw=1,
        label=f'$\\beta/\\nu$ = {beta_nu:.3f}({int(beta_nu_err*1000):d})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('L'); ax.set_ylabel('$M(T_c)$')
ax.legend(fontsize=7)
label_panel(ax, 'b')

# (c) 1/nu from |dU/dT|_max ~ L^(1/nu)
ax = axes[2]
logDU = np.log(valid['dU_dT_min'])
p3, cov3 = np.polyfit(logL, logDU, 1, cov=True)
inv_nu = p3[0]
inv_nu_err = np.sqrt(cov3[0, 0])
nu_fit = 1.0 / inv_nu if inv_nu > 0 else np.nan
nu_fit_err = inv_nu_err / inv_nu**2 if inv_nu > 0 else np.nan

ax.plot(valid['L'], valid['dU_dT_min'], 'ko', ms=5)
ax.plot(L_fit, np.exp(p3[1]) * L_fit**p3[0], 'r-', lw=1,
        label=f'$1/\\nu$ = {inv_nu:.3f}({int(inv_nu_err*1000):d})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('L'); ax.set_ylabel('$|dU_4/dT|_{\\max}$')
ax.legend(fontsize=7)
label_panel(ax, 'c')

plt.tight_layout()
plt.savefig(FIGDIR / 'fig6_exponent_scaling.pdf')
plt.show()

# Summary table
print("\n" + "="*60)
print("CRITICAL EXPONENT RESULTS")
print("="*60)
print(f"{'Quantity':<25} {'Measured':>12} {'Theory':>10} {'Error':>8}")
print("-"*60)
print(f"{'Tc (Binder)':<25} {Tc_use:>12.4f} {TC_3D:>10.4f} {abs(Tc_use-TC_3D)/TC_3D*100:>7.2f}%")
print(f"{'gamma/nu (chi peak)':<25} {gamma_nu:>12.3f} {GAMMA_3D/NU_3D:>10.3f} {abs(gamma_nu-GAMMA_3D/NU_3D)/(GAMMA_3D/NU_3D)*100:>7.1f}%")
print(f"{'beta/nu (M at Tc)':<25} {beta_nu:>12.3f} {BETA_3D/NU_3D:>10.3f} {abs(beta_nu-BETA_3D/NU_3D)/(BETA_3D/NU_3D)*100:>7.1f}%")
print(f"{'1/nu (dU/dT peak)':<25} {inv_nu:>12.3f} {1/NU_3D:>10.3f} {abs(inv_nu-1/NU_3D)/(1/NU_3D)*100:>7.1f}%")
print(f"{'nu':<25} {nu_fit:>12.3f} {NU_3D:>10.3f} {abs(nu_fit-NU_3D)/NU_3D*100:>7.1f}%")
# Hyperscaling check
hs = 2*beta_nu + gamma_nu
print(f"\n{'2*beta/nu + gamma/nu':<25} {hs:>12.3f} {'3.0 (d=3)':>10} {abs(hs-3.0)/3.0*100:>7.1f}%")

### 2e. Scaling Collapse

Test data collapse of $\chi L^{-\gamma/\nu}$ vs $(T - T_c) L^{1/\nu}$ using both
literature exponents and our fitted values.

In [ ]:
# Scaling collapse
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# Use FSS sweep data (wider T range)
for ax_idx, (label, tc, nu, gn) in enumerate([
    ('Literature exponents', TC_3D, NU_3D, GAMMA_3D/NU_3D),
    ('Fitted exponents', Tc_use, nu_fit, gamma_nu),
]):
    ax = axes[ax_idx]
    for N in sorted(fss.keys()):
        df = fss[N]
        V = N**3
        chi_conn = (1.0 / df['T']) * V * (df['M2'] - df['M']**2)
        
        x = (df['T'] - tc) * N**(1/nu)
        y = chi_conn * N**(-gn)
        
        c = COLORS[sorted(fss.keys()).index(N) % len(COLORS)]
        mk = SIZE_MARKERS.get(N, 'o')
        ax.plot(x, y, mk, ms=3, color=c, label=f'L={N}')
    
    ax.set_xlabel(f'$(T - T_c) L^{{1/\\nu}}$')
    ax.set_ylabel(f'$\\chi L^{{-\\gamma/\\nu}}$')
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=6)
    ax.set_xlim(-20, 20)
    label_panel(ax, chr(ord('a') + ax_idx))

plt.tight_layout()
plt.savefig(FIGDIR / 'fig7_scaling_collapse.pdf')
plt.show()

---
## 3. J-Fitting: BCC Iron and FCC Nickel

Map simulation $T_c/J$ to experimental Curie temperatures to extract exchange coupling $J$:
- **BCC Fe**: $T_c^{\exp} = 1043$ K, coordination $z=8$
- **FCC Ni**: $T_c^{\exp} = 627$ K, coordination $z=12$

Mean-field prediction: $T_c^{MF} = zJ/3k_B$ (overestimates by ~50%)

We find $T_c/J$ from the Binder crossing on our BCC/FCC mesh simulations,
then solve $J = k_B T_c^{\exp} / (T_c/J)_{\text{sim}}$.

In [ ]:
# Load J-fit data
jfit_dir = DATADIR / 'jfit'
bcc_data = {}
fcc_data = {}
for f in sorted(jfit_dir.glob('bcc_iron_N*_sweep.csv')):
    N = int(f.stem.split('N')[1].split('_')[0])
    bcc_data[N] = pd.read_csv(f)
for f in sorted(jfit_dir.glob('fcc_nickel_N*_sweep.csv')):
    N = int(f.stem.split('N')[1].split('_')[0])
    fcc_data[N] = pd.read_csv(f)

print(f"BCC Fe sizes: {sorted(bcc_data.keys())}")
print(f"FCC Ni sizes: {sorted(fcc_data.keys())}")

# Physical constants
kB = 8.617333e-5  # eV/K (Boltzmann constant)
TC_FE_EXP = 1043  # K
TC_NI_EXP = 627   # K

fig, axes = plt.subplots(2, 3, figsize=(10, 6))

# --- BCC Iron ---
# (a) M(T) curves
ax = axes[0, 0]
bcc_binder_interps = {}
for N in sorted(bcc_data.keys()):
    df = bcc_data[N]
    c = COLORS[sorted(bcc_data.keys()).index(N) % len(COLORS)]
    mk = SIZE_MARKERS.get(N, 'o')
    ax.plot(df['T'], df['M'], mk+'-', ms=3, lw=0.8, color=c, label=f'N={N}')
ax.set_xlabel('T / J'); ax.set_ylabel('|m|')
ax.set_title('BCC Iron', fontsize=9)
ax.legend(fontsize=6)
label_panel(ax, 'a')

# (b) Binder cumulant
ax = axes[0, 1]
for N in sorted(bcc_data.keys()):
    df = bcc_data[N]
    U = binder(df['M2'], df['M4'])
    c = COLORS[sorted(bcc_data.keys()).index(N) % len(COLORS)]
    mk = SIZE_MARKERS.get(N, 'o')
    ax.plot(df['T'], U, mk+'-', ms=3, lw=0.8, color=c, label=f'N={N}')
    bcc_binder_interps[N] = interp1d(df['T'].values, U.values, kind='cubic', fill_value='extrapolate')
ax.set_xlabel('T / J'); ax.set_ylabel('$U_4$')
ax.axhline(2/3, color='gray', ls=':', lw=0.5)
ax.legend(fontsize=6)
label_panel(ax, 'b')

# Find BCC Tc/J from largest pair crossing
bcc_sizes = sorted(bcc_data.keys())
bcc_crossings = []
for i in range(len(bcc_sizes)):
    for j in range(i+1, len(bcc_sizes)):
        L1, L2 = bcc_sizes[i], bcc_sizes[j]
        try:
            Tc_bcc = brentq(lambda T: bcc_binder_interps[L1](T) - bcc_binder_interps[L2](T), 4.0, 8.0)
            bcc_crossings.append((L1, L2, Tc_bcc))
        except (ValueError, Exception):
            pass

Tc_bcc_sim = np.mean([c[2] for c in bcc_crossings]) if bcc_crossings else 6.35
J_fe = kB * TC_FE_EXP / Tc_bcc_sim  # eV
J_fe_meV = J_fe * 1000

# (c) J extraction summary
ax = axes[0, 2]
ax.axis('off')
txt = f"BCC Iron Results:\n\n"
txt += f"$T_c/J$ (sim) = {Tc_bcc_sim:.3f}\n"
txt += f"$T_c^{{\\exp}}$ = {TC_FE_EXP} K\n\n"
txt += f"$J_{{Fe}}$ = {J_fe_meV:.1f} meV\n"
txt += f"$J_{{Fe}}$ = {J_fe*11604:.0f} K\n\n"
txt += f"Mean-field: $J_{{MF}}$ = {kB*TC_FE_EXP*3/8*1000:.1f} meV\n"
txt += f"(z=8)"
ax.text(0.1, 0.9, txt, transform=ax.transAxes, fontsize=8, va='top', family='monospace')
label_panel(ax, 'c')

# --- FCC Nickel ---
ax = axes[1, 0]
fcc_binder_interps = {}
for N in sorted(fcc_data.keys()):
    df = fcc_data[N]
    c = COLORS[sorted(fcc_data.keys()).index(N) % len(COLORS)]
    mk = SIZE_MARKERS.get(N, 'o')
    ax.plot(df['T'], df['M'], mk+'-', ms=3, lw=0.8, color=c, label=f'N={N}')
ax.set_xlabel('T / J'); ax.set_ylabel('|m|')
ax.set_title('FCC Nickel', fontsize=9)
ax.legend(fontsize=6)
label_panel(ax, 'd')

ax = axes[1, 1]
for N in sorted(fcc_data.keys()):
    df = fcc_data[N]
    U = binder(df['M2'], df['M4'])
    c = COLORS[sorted(fcc_data.keys()).index(N) % len(COLORS)]
    mk = SIZE_MARKERS.get(N, 'o')
    ax.plot(df['T'], U, mk+'-', ms=3, lw=0.8, color=c, label=f'N={N}')
    fcc_binder_interps[N] = interp1d(df['T'].values, U.values, kind='cubic', fill_value='extrapolate')
ax.set_xlabel('T / J'); ax.set_ylabel('$U_4$')
ax.axhline(2/3, color='gray', ls=':', lw=0.5)
ax.legend(fontsize=6)
label_panel(ax, 'e')

# FCC Tc
fcc_sizes = sorted(fcc_data.keys())
fcc_crossings = []
for i in range(len(fcc_sizes)):
    for j in range(i+1, len(fcc_sizes)):
        L1, L2 = fcc_sizes[i], fcc_sizes[j]
        try:
            Tc_fcc = brentq(lambda T: fcc_binder_interps[L1](T) - fcc_binder_interps[L2](T), 7.0, 12.0)
            fcc_crossings.append((L1, L2, Tc_fcc))
        except (ValueError, Exception):
            pass

Tc_fcc_sim = np.mean([c[2] for c in fcc_crossings]) if fcc_crossings else 9.79
J_ni = kB * TC_NI_EXP / Tc_fcc_sim
J_ni_meV = J_ni * 1000

ax = axes[1, 2]
ax.axis('off')
txt = f"FCC Nickel Results:\n\n"
txt += f"$T_c/J$ (sim) = {Tc_fcc_sim:.3f}\n"
txt += f"$T_c^{{\\exp}}$ = {TC_NI_EXP} K\n\n"
txt += f"$J_{{Ni}}$ = {J_ni_meV:.1f} meV\n"
txt += f"$J_{{Ni}}$ = {J_ni*11604:.0f} K\n\n"
txt += f"Mean-field: $J_{{MF}}$ = {kB*TC_NI_EXP*3/12*1000:.1f} meV\n"
txt += f"(z=12)"
ax.text(0.1, 0.9, txt, transform=ax.transAxes, fontsize=8, va='top', family='monospace')
label_panel(ax, 'f')

fig.suptitle('Exchange Coupling Extraction from Crystal Mesh Simulations', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(FIGDIR / 'fig8_jfit_bcc_fcc.pdf')
plt.show()

print("\nBCC Fe crossings:")
for L1, L2, Tc in bcc_crossings:
    print(f"  N={L1} x N={L2}: Tc/J = {Tc:.3f}")
print(f"  Mean Tc/J = {Tc_bcc_sim:.3f}, J_Fe = {J_fe_meV:.1f} meV")

print(f"\nFCC Ni crossings:")
for L1, L2, Tc in fcc_crossings:
    print(f"  N={L1} x N={L2}: Tc/J = {Tc:.3f}")
print(f"  Mean Tc/J = {Tc_fcc_sim:.3f}, J_Ni = {J_ni_meV:.1f} meV")

---
## 4. Kibble-Zurek Mechanism

The Kibble-Zurek mechanism predicts the density of topological defects
after a symmetry-breaking quench scales as a power law of the quench rate:

$$\rho \sim \tau_Q^{-d\nu/(1+z\nu)}$$

For the 3D Ising model ($d=3$, $\nu=0.630$, $z \approx 2$):
$$\rho \sim \tau_Q^{-0.86}$$

We quench linearly from $T = 6.0 > T_c$ to $T = 1.0 < T_c$ over $\tau_Q$ Metropolis sweeps
and measure the domain wall density at the end.

In [ ]:
# Kibble-Zurek analysis
kz_files = sorted((DATADIR / 'kz').glob('kz_N*.csv'))
print(f"KZ data files: {[f.name for f in kz_files]}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

kz_results = {}
for f in kz_files:
    N = int(f.stem.split('N')[1])
    df = pd.read_csv(f)
    kz_results[N] = df
    
    # Filter out zero-rho points (can't take log)
    mask = df['rho'] > 0
    if mask.sum() < 2:
        print(f"  N={N}: only {mask.sum()} non-zero rho points -- need larger lattice or more trials")
        axes[0].plot(df.loc[mask, 'tau_q'], df.loc[mask, 'rho'], 'o', ms=4, label=f'N={N}')
        continue
    
    df_pos = df[mask]
    axes[0].plot(df_pos['tau_q'], df_pos['rho'], 'o-', ms=4, lw=0.8, label=f'N={N}')
    
    # Log-log fit
    log_tau = np.log(df_pos['tau_q'])
    log_rho = np.log(df_pos['rho'])
    
    if len(log_tau) >= 2:
        p, cov = np.polyfit(log_tau, log_rho, 1, cov=True)
        slope = p[0]
        slope_err = np.sqrt(cov[0, 0])
        
        tau_fit = np.logspace(np.log10(df_pos['tau_q'].min()), np.log10(df_pos['tau_q'].max()), 50)
        axes[0].plot(tau_fit, np.exp(p[1]) * tau_fit**p[0], '--', lw=1,
                    label=f'slope = {slope:.2f}({int(slope_err*100):d})')
        
        kz_results[N] = (df, slope, slope_err)

axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('$\\tau_Q$ (sweeps)')
axes[0].set_ylabel('$\\rho$ (domain wall density)')
axes[0].legend(fontsize=6)
axes[0].set_title('KZ: Defect Density vs Quench Time')
label_panel(axes[0], 'a')

# KZ theory line
ax = axes[1]
# Expected exponent for 3D Ising with Metropolis dynamics (z~2)
nu_kz = 0.630
z_dyn = 2.0
d = 3
kz_exp = -d * nu_kz / (1 + z_dyn * nu_kz)
ax.axhline(kz_exp, color='red', ls='--', lw=1, label=f'Theory: {kz_exp:.3f}')

for N, val in kz_results.items():
    if isinstance(val, tuple):
        df, slope, slope_err = val
        ax.errorbar(N, slope, yerr=slope_err, fmt='ko', ms=6, capsize=4)
        ax.annotate(f'{slope:.2f}', (N, slope), textcoords='offset points', 
                   xytext=(10, 5), fontsize=8)

ax.set_xlabel('Lattice size N')
ax.set_ylabel('KZ exponent (slope)')
ax.legend(fontsize=7)
ax.set_title('KZ Exponent Comparison')
label_panel(ax, 'b')

plt.tight_layout()
plt.savefig(FIGDIR / 'fig9_kibble_zurek.pdf')
plt.show()

print(f"\nKZ theory prediction: rho ~ tau_Q^{{{kz_exp:.3f}}}")
print("Note: Quick mode uses small lattice (N=20) with few trials.")
print("Full run (N=80, 50 trials) needed for reliable KZ scaling.")

---
## 5. Domain Coarsening

After an instantaneous quench from $T \gg T_c$ to $T_q < T_c$, domains grow with the
Allen-Cahn power law:

$$\rho(t) \sim t^{-1/z_{\text{growth}}}$$

where $z_{\text{growth}} = 2$ for non-conserved order parameter (Model A dynamics),
giving $\rho \sim t^{-1/2}$ in the late-time regime.

In [ ]:
# Domain coarsening analysis
coarsen_files = sorted((DATADIR / 'coarsening').glob('coarsening_N*.csv'))
print(f"Coarsening files: {[f.name for f in coarsen_files]}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

coarsen_fits = []
for f in coarsen_files:
    parts = f.stem.split('_')
    N = int(parts[1].replace('N', ''))
    T_q = float(parts[2].replace('T', ''))
    df = pd.read_csv(f)
    
    c = COLORS[coarsen_files.index(f) % len(COLORS)]
    
    # (a) rho vs t (log-log)
    mask = df['t'] > 0  # skip t=0
    axes[0].plot(df.loc[mask, 't'], df.loc[mask, 'rho'], '-', lw=0.8, color=c,
                label=f'N={N}, $T_q$={T_q}')
    
    # Fit power law in late time regime (last 50% of data)
    df_late = df[mask].iloc[len(df[mask])//2:]
    df_late = df_late[df_late['rho'] > 0]
    
    if len(df_late) >= 5:
        log_t = np.log(df_late['t'].values)
        log_rho = np.log(df_late['rho'].values)
        p, cov = np.polyfit(log_t, log_rho, 1, cov=True)
        slope = p[0]
        slope_err = np.sqrt(cov[0, 0])
        coarsen_fits.append(dict(N=N, T_q=T_q, slope=slope, slope_err=slope_err))
        
        t_fit = np.logspace(np.log10(df_late['t'].min()), np.log10(df_late['t'].max()), 50)
        axes[0].plot(t_fit, np.exp(p[1]) * t_fit**p[0], '--', lw=1.2, color=c)

axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('t (sweeps)')
axes[0].set_ylabel('$\\rho$ (domain wall density)')
axes[0].legend(fontsize=6)
label_panel(axes[0], 'a')

# Theory reference line
t_ref = np.logspace(2, 4, 50)
axes[0].plot(t_ref, 0.3 * t_ref**(-0.5), 'k:', lw=0.8, alpha=0.5, label='$t^{-1/2}$')
axes[0].legend(fontsize=6)

# (b) Exponent summary
ax = axes[1]
if coarsen_fits:
    for fit in coarsen_fits:
        ax.errorbar(fit['N'], fit['slope'], yerr=fit['slope_err'], 
                    fmt='ko', ms=6, capsize=4)
        ax.annotate(f"{fit['slope']:.3f}", (fit['N'], fit['slope']),
                   textcoords='offset points', xytext=(10, 5), fontsize=8)

ax.axhline(-0.5, color='red', ls='--', lw=1, label='Allen-Cahn ($-1/2$)')
ax.axhline(-1/3, color='blue', ls=':', lw=1, label='$-1/3$ (early time)')
ax.set_xlabel('Lattice size N')
ax.set_ylabel('Coarsening exponent')
ax.legend(fontsize=7)
label_panel(ax, 'b')

plt.tight_layout()
plt.savefig(FIGDIR / 'fig10_domain_coarsening.pdf')
plt.show()

print("\nCoarsening fit results:")
for fit in coarsen_fits:
    print(f"  N={fit['N']}, T_q={fit['T_q']}: slope = {fit['slope']:.3f} +/- {fit['slope_err']:.3f}")
print("Allen-Cahn prediction: -0.500 (Model A, non-conserved OP)")

---
## 6. Results Summary

Comprehensive table of all measured quantities, uncertainties, and comparison to theory.

In [ ]:
# ================================================================
# COMPREHENSIVE RESULTS SUMMARY
# ================================================================

results = []

# 1. FSS results
results.append(dict(
    Category='FSS (3D Cubic)',
    Quantity='$T_c$',
    Measured=f'{Tc_use:.4f}',
    Theory=f'{TC_3D:.4f}',
    Error_pct=f'{abs(Tc_use-TC_3D)/TC_3D*100:.2f}%',
    Method='Binder crossing'
))
results.append(dict(
    Category='FSS (3D Cubic)',
    Quantity='$\\gamma/\\nu$',
    Measured=f'{gamma_nu:.3f}({int(gamma_nu_err*1000)})',
    Theory=f'{GAMMA_3D/NU_3D:.3f}',
    Error_pct=f'{abs(gamma_nu-GAMMA_3D/NU_3D)/(GAMMA_3D/NU_3D)*100:.1f}%',
    Method='$\\chi_{\\max}$ peak scaling'
))
results.append(dict(
    Category='FSS (3D Cubic)',
    Quantity='$\\beta/\\nu$',
    Measured=f'{beta_nu:.3f}({int(beta_nu_err*1000)})',
    Theory=f'{BETA_3D/NU_3D:.3f}',
    Error_pct=f'{abs(beta_nu-BETA_3D/NU_3D)/(BETA_3D/NU_3D)*100:.1f}%',
    Method='$M(T_c)$ scaling'
))
results.append(dict(
    Category='FSS (3D Cubic)',
    Quantity='$\\nu$',
    Measured=f'{nu_fit:.3f}({int(nu_fit_err*1000) if not np.isnan(nu_fit_err) else "?"})',
    Theory=f'{NU_3D:.3f}',
    Error_pct=f'{abs(nu_fit-NU_3D)/NU_3D*100:.1f}%' if not np.isnan(nu_fit) else 'N/A',
    Method='$dU_4/dT$ peak scaling'
))
results.append(dict(
    Category='FSS (3D Cubic)',
    Quantity='$2\\beta/\\nu + \\gamma/\\nu$',
    Measured=f'{2*beta_nu + gamma_nu:.3f}',
    Theory='3.000',
    Error_pct=f'{abs(2*beta_nu+gamma_nu-3)/3*100:.1f}%',
    Method='Hyperscaling check ($d=3$)'
))

# 2. J-fitting
results.append(dict(
    Category='J-Fitting',
    Quantity='$T_c/J$ (BCC)',
    Measured=f'{Tc_bcc_sim:.3f}',
    Theory='6.355 (literature)',
    Error_pct='--',
    Method='Binder crossing (mesh)'
))
results.append(dict(
    Category='J-Fitting',
    Quantity='$J_{{Fe}}$',
    Measured=f'{J_fe_meV:.1f} meV',
    Theory='~14 meV (DFT)',
    Error_pct='--',
    Method=f'$k_B T_c^{{exp}} / (T_c/J)$'
))
results.append(dict(
    Category='J-Fitting',
    Quantity='$T_c/J$ (FCC)',
    Measured=f'{Tc_fcc_sim:.3f}',
    Theory='9.79 (literature)',
    Error_pct='--',
    Method='Binder crossing (mesh)'
))
results.append(dict(
    Category='J-Fitting',
    Quantity='$J_{{Ni}}$',
    Measured=f'{J_ni_meV:.1f} meV',
    Theory='~8 meV (DFT)',
    Error_pct='--',
    Method=f'$k_B T_c^{{exp}} / (T_c/J)$'
))

# 3. Coarsening
for fit in coarsen_fits:
    results.append(dict(
        Category='Coarsening',
        Quantity=f'exponent (N={fit["N"]})',
        Measured=f'{fit["slope"]:.3f}({int(fit["slope_err"]*1000)})',
        Theory='-0.500',
        Error_pct=f'{abs(fit["slope"]+0.5)/0.5*100:.1f}%',
        Method='Allen-Cahn $\\rho \\sim t^{-1/2}$'
    ))

# Display
summary_df = pd.DataFrame(results)
print("="*90)
print("V2 RESULTS SUMMARY")
print("="*90)
for _, row in summary_df.iterrows():
    print(f"  {row['Category']:20s} | {row['Quantity']:25s} | {row['Measured']:>15s} | {row['Theory']:>15s} | {row['Error_pct']:>8s}")
print("="*90)

# Save to markdown
with open('results_summary.md', 'w') as f:
    f.write("# V2 Ising Model Results Summary\n\n")
    f.write(f"Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write("## Key Results\n\n")
    f.write("| Category | Quantity | Measured | Theory | Error | Method |\n")
    f.write("|---|---|---|---|---|---|\n")
    for _, row in summary_df.iterrows():
        f.write(f"| {row['Category']} | {row['Quantity']} | {row['Measured']} | {row['Theory']} | {row['Error_pct']} | {row['Method']} |\n")
    f.write("\n## Notes\n\n")
    f.write("- Connected susceptibility: chi = beta * V * (<|m|^2> - <|m|>^2)\n")
    f.write("- Wolff cluster algorithm used for all equilibrium measurements\n")
    f.write("- Metropolis sweeps used for KZ quench dynamics\n")
    f.write("- Histogram reweighting (Ferrenberg-Swendsen) for fine T resolution near Tc\n")
    f.write("- Jackknife error estimation (10-20 blocks) on all fitted quantities\n")

print("\nSaved results_summary.md")